# NARA - Evaluasi acuan revisi

**Workflow:** baca fungsi dan sumber → tinjau kode/hasil tersimpan → jalankan hanya jika memang memerlukan eksperimen baru.

Evaluasi 79 narasi sudah dihitung dan diperiksa. Menjalankan notebook ini hanya reproduksi opsional; UJI-019 tetap DEFERRED. Tidak ada training atau inferensi baru.

Kode sumber seluruh sel dipertahankan. Nama TanggapMakna pada path atau metadata historis tetap menunjukkan asal aset. File ini disusun sebagai lampiran kode; tidak wajib dijalankan ulang.


# SLR09 — Evaluasi NARA dengan acuan revisi

Workflow: validasi snapshot Excel → gunakan hanya APPROVED lengkap → cocokkan prediksi run tetap → hitung metrik dan confusion matrix → ekspor hasil.

Snapshot ini berisi **79 APPROVED / 1.701 token** dan **UJI-019 DEFERRED / 17 token**. Hasil adalah evaluasi parsial dari sampel 80, dengan status review tetap. Notebook tidak mengubah label, menjalankan training, memuat bobot, atau menjalankan inferensi neural baru. SENA dan tujuh field adapter tidak dievaluasi di notebook ini.

Hasil yang telah dihitung dalam pemeriksaan: macro F1 non-O **53,43% → 79,32%**, F1 span tepat **43,69% → 63,84%**. Gunakan CPU; GPU tidak diperlukan. Run all dipakai untuk mereproduksi hasil.


In [ ]:
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path, PurePosixPath

EVAL_ROOT = Path(os.environ.get("SLR09_EVAL_ROOT", "/content/slr09_nara_eval_revised_v01" if Path("/content").is_dir() else "slr09_nara_eval_runtime")).resolve()
PAYLOAD_B64 = "..."
PAYLOAD_SHA256 = "028d8abd49ed46f773fd2c1911c201937319060bb894699bcf8d792d77055fe6"
raw = base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(raw).hexdigest() != PAYLOAD_SHA256:
    raise ValueError("Payload notebook berubah")
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for item in archive.infolist():
        relative = PurePosixPath(item.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Path snapshot tidak sah")
        destination = EVAL_ROOT.joinpath(*relative.parts)
        content = archive.read(item)
        if destination.exists() and destination.read_bytes() != content:
            raise ValueError("Snapshot lokal berbeda; gunakan direktori SLR09_EVAL_ROOT yang baru: " + str(destination))
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
sys.path.insert(0, str(EVAL_ROOT / "code"))
print("Snapshot input dan kode siap. Tidak ada bobot model dalam payload.")


## Sumber prediksi

ZIP hasil run `20261018T001122_607337Z` sudah ada di folder Drive `SILAR_SLR09`. Hash arsip diperiksa sebelum evaluasi. Bila menjalankan dari paket kode lengkap di komputer lokal, file pada folder sources dipakai. Untuk lokasi lain, set environment `SLR09_RESULTS_ZIP` ke path ZIP tersebut.


In [ ]:
REPORT_NAME = "SLR09_NARA_Hasil_FineTune_20261018T001122_607337Z.zip"
configured = os.environ.get("SLR09_RESULTS_ZIP")
local_source = Path.cwd() / "sources" / REPORT_NAME
if configured:
    RESULTS_ZIP = Path(configured)
elif local_source.is_file():
    RESULTS_ZIP = local_source
else:
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError("ZIP hasil diperlukan. Set SLR09_RESULTS_ZIP atau jalankan dari folder paket kode lengkap.")
    drive.mount("/content/drive")
    RESULTS_ZIP = Path("/content/drive/MyDrive/SILAR_SLR09") / REPORT_NAME
if not RESULTS_ZIP.is_file():
    raise FileNotFoundError("ZIP hasil run belum ditemukan: " + str(RESULTS_ZIP))
REVIEW_XLSX = EVAL_ROOT / "sources/SLR09_NARA_Review_Uji_80_Reviewed.xlsx"
RESULT_DIR = EVAL_ROOT / "results"
EVALUATION_SCOPE = "approved-subset"
print("Scope:", EVALUATION_SCOPE, "— kasus DEFERRED tetap tidak diberi skor.")


## Validasi dan evaluasi

Validator asli tetap menolak evaluasi penuh 80 selama ada DEFERRED. Analisis berikut secara eksplisit memakai subset APPROVED, bukan mengubah gate tersebut. Label sementara UJI-019 tidak dimasukkan.


In [ ]:
from validate_review import validate_file
from evaluate_reviewed_nara import evaluate, plot
validation, _, _, _ = validate_file(REVIEW_XLSX, EVAL_ROOT / "data")
print(json.dumps({k:validation[k] for k in ["ready_narratives", "deferred_narratives", "missing_labels", "invalid_labels", "ready_to_evaluate"]}, indent=2))
result = evaluate(REVIEW_XLSX, EVAL_ROOT / "data", RESULTS_ZIP,
                  EVAL_ROOT / "sources/historical_test_original_labels.jsonl",
                  RESULT_DIR, scope=EVALUATION_SCOPE)
plot(result, RESULT_DIR)
print(json.dumps({"evaluation_scope":result["evaluation_scope"], "coverage":result["coverage"]}, indent=2))
for model in ["baseline", "finetuned"]:
    print(model, json.dumps(result["metrics"][model]["reviewed_gold"]["token_metrics"], indent=2))


## Laporan, matriks, dan batas interpretasi

Macro F1 menggunakan 14 label non-O yang tetap; O ikut memengaruhi false positive/negative. F1 span memerlukan batas dan label yang sama persis, bukan evaluasi BIO. Hasil WOUND memiliki dukungan sangat kecil. Test historis 1.000 narasi tetap dilaporkan dengan penurunannya; evaluasi parsial ini tidak menggantikan hasil tersebut.


In [ ]:
from write_report import write_report
REPORT_PATH = EVAL_ROOT / "SLR09_NARA_Laporan_Evaluasi_Acuan_Revisi_v0.1.md"
write_report(result, REPORT_PATH, RESULT_DIR)
try:
    from IPython.display import display, Markdown, Image
except ImportError:
    print(REPORT_PATH.read_text(encoding="utf-8"))
else:
    display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))
    for filename in ["baseline_confusion_matrix.png", "finetuned_confusion_matrix.png", "f1_per_label.png"]:
        display(Image(filename=str(RESULT_DIR / filename)))


## Ekspor hasil kecil

Ekspor berisi metrik, cakupan 79/80, status satu kasus tertunda, acuan subset yang disetujui, rincian kesalahan, grafik, dan laporan. Tidak membuat gold penuh 80 atau mengubah workbook input.


In [ ]:
import shutil
from datetime import datetime, timezone
name = "SLR09_NARA_Hasil_Evaluasi_Acuan_Revisi_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + ".zip"
export_zip = EVAL_ROOT / name
with zipfile.ZipFile(export_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(RESULT_DIR.rglob("*")):
        if file.is_file():
            archive.write(file, "results/" + file.relative_to(RESULT_DIR).as_posix())
    archive.write(REPORT_PATH, REPORT_PATH.name)
try:
    from google.colab import files
except ImportError:
    print("Hasil:", export_zip)
else:
    drive_folder = Path("/content/drive/MyDrive/SILAR_SLR09")
    if drive_folder.is_dir():
        shutil.copyfile(export_zip, drive_folder / name)
    files.download(str(export_zip))


## Kode sumber lengkap

Empat modul berikut juga ditampilkan sebagai teks yang dapat dibaca. Isinya sama dengan modul yang disimpan bootstrap di folder code. Paket ZIP menyediakan file Python terpisah beserta pembuat notebook dan pemeriksaan independen.


### validate_review.py

```python
#!/usr/bin/env python3
"""Validasi Excel review tanpa mengubah label kosong menjadi O.

Hanya pustaka standar Python. Tidak menjalankan model atau menghitung metrik
sebelum semua 80 narasi disetujui. --freeze menulis gold hanya bila lengkap.
"""
from __future__ import annotations

import argparse
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import posixpath
import re
import xml.etree.ElementTree as ET
import zipfile

NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
RID = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def read_xlsx(path):
    """Read cell values/formula flags; never evaluate formulas or external links."""
    sheets = {}
    with zipfile.ZipFile(path) as archive:
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Nama anggota ZIP Excel duplikat")
        if archive.testzip() is not None:
            raise ValueError("ZIP Excel rusak")
        shared = []
        if "xl/sharedStrings.xml" in archive.namelist():
            root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            shared = ["".join(t.text or "" for t in si.findall(".//m:t", NS)) for si in root.findall("m:si", NS)]
        rels = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        targets = {r.attrib["Id"]: r.attrib["Target"] for r in rels}
        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        for sheet in workbook.findall("m:sheets/m:sheet", NS):
            target = targets[sheet.attrib[RID]]
            member = target.lstrip("/") if target.startswith("/") else posixpath.normpath("xl/" + target)
            root = ET.fromstring(archive.read(member))
            values = {}
            for cell in root.findall(".//m:sheetData/m:row/m:c", NS):
                kind, ref = cell.attrib.get("t"), cell.attrib["r"]
                node = cell.find("m:v", NS)
                text = node.text if node is not None and node.text is not None else ""
                if kind == "s":
                    text = shared[int(text)]
                elif kind == "inlineStr":
                    text = "".join(t.text or "" for t in cell.findall(".//m:t", NS))
                values[ref] = {"value": text, "formula": cell.find("m:f", NS) is not None, "type": kind}
            sheets[sheet.attrib["name"]] = values
    return sheets


def cell_value(sheet, ref, *, allow_formula=False):
    cell = sheet.get(ref, {"value": "", "formula": False})
    if cell["formula"] and not allow_formula:
        raise ValueError("Rumus tidak diizinkan pada input/sumber: " + ref)
    return cell["value"]


def load_locked_data(data_dir):
    data_dir = Path(data_dir)
    lock = read_json(data_dir / "data_lock.json")
    for name, expected in lock.items():
        if digest(data_dir / name) != expected:
            raise ValueError("Sumber paket berubah: " + name)
    records = read_json(data_dir / "review_records.json")["records"]
    schema = read_json(data_dir / "label_schema.json")
    if len(records) != 80 or sum(len(r["tokens"]) for r in records) != 1718:
        raise ValueError("Jumlah narasi/token berubah")
    if schema["blank_label_is_o"] or len(schema["labels"]) != 15:
        raise ValueError("Skema label tidak sesuai")
    return records, schema


def read_annotations(sheets, records, schema):
    for name in ("Narasi", "Token", "Pedoman"):
        if name not in sheets:
            raise ValueError("Tab wajib tidak ditemukan: " + name)
    guide = sheets["Pedoman"]
    if [cell_value(guide, f"C{r}") for r in range(8, 23)] != [r["label"] for r in schema["labels"]]:
        raise ValueError("Daftar 15 label di Pedoman berubah")
    narrative_sheet, token_sheet = sheets["Narasi"], sheets["Token"]
    reviewer = str(cell_value(narrative_sheet, "C4")).strip()
    narratives, tokens = {}, {}
    # Read populated record rows independent of row order, so whole-table
    # sorting does not move notes or labels away from their stable keys.
    narrative_rows = sorted({int(m.group(2)) for ref in narrative_sheet if (m := re.fullmatch(r"([B-DGH])(\d+)", ref)) and int(m.group(2)) >= 8})
    for n in narrative_rows:
        key = cell_value(narrative_sheet, f"B{n}")
        if not key:
            if any(cell_value(narrative_sheet, f"{c}{n}") for c in ("C", "G", "H")):
                raise ValueError(f"ID narasi kosong pada baris {n}")
            continue
        if key in narratives:
            raise ValueError("ID narasi duplikat: " + key)
        narratives[key] = {"review_id": key, "text": cell_value(narrative_sheet, f"C{n}"), "token_count": cell_value(narrative_sheet, f"D{n}"), "status": cell_value(narrative_sheet, f"G{n}"), "note": cell_value(narrative_sheet, f"H{n}")}
    token_rows = sorted({int(m.group(2)) for ref in token_sheet if (m := re.fullmatch(r"([B-F])(\d+)", ref)) and int(m.group(2)) >= 8})
    for n in token_rows:
        key = cell_value(token_sheet, f"B{n}")
        if not key:
            if any(cell_value(token_sheet, f"{c}{n}") for c in ("C", "D", "E", "F")):
                raise ValueError(f"ID token kosong pada baris {n}")
            continue
        position = cell_value(token_sheet, f"C{n}")
        try:
            position_number = int(position)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Posisi token tidak valid pada baris {n}") from exc
        stable_key = (key, position_number)
        if stable_key in tokens:
            raise ValueError("Kunci token duplikat: " + str(stable_key))
        tokens[stable_key] = {"token": cell_value(token_sheet, f"D{n}"), "label": cell_value(token_sheet, f"E{n}"), "note": cell_value(token_sheet, f"F{n}"), "excel_row": n}
    expected_ids = {r["review_id"] for r in records}
    expected_keys = {(r["review_id"], i + 1) for r in records for i in range(len(r["tokens"]))}
    if set(narratives) != expected_ids or set(tokens) != expected_keys:
        raise ValueError("ID/jumlah/posisi narasi atau token berubah")
    for row in records:
        if narratives[row["review_id"]]["text"] != row["text"]:
            raise ValueError("Teks narasi berubah: " + row["review_id"])
        if str(narratives[row["review_id"]]["token_count"]) != str(len(row["tokens"])):
            raise ValueError("Jumlah token sumber berubah: " + row["review_id"])
        for i, token in enumerate(row["tokens"], 1):
            if tokens[(row["review_id"], i)]["token"] != token:
                raise ValueError(f"Token sumber berubah: {row['review_id']} posisi {i}")
    return reviewer, narratives, tokens


def validate_annotations(records, schema, reviewer, narratives, tokens):
    allowed = {r["label"] for r in schema["labels"]}
    counts, details, canonicalizations, invalid = Counter(), [], [], []
    labels_by_id = {}
    for row in records:
        key = row["review_id"]
        status_raw = narratives[key]["status"]
        status = str(status_raw).upper()
        note = str(narratives[key]["note"]).strip()
        missing, bad, labels = 0, 0, []
        for position in range(1, len(row["tokens"]) + 1):
            raw = tokens[(key, position)]["label"]
            if raw is None or raw == "":
                missing += 1
                labels.append(None)
                continue
            label = str(raw).upper()  # Case only; no guessed aliases/whitespace/O fill.
            if label not in allowed:
                bad += 1
                invalid.append({"review_id": key, "position": position, "label": raw})
            elif raw != label:
                canonicalizations.append({"review_id": key, "position": position, "input": raw, "canonical_label": label})
            labels.append(label)
        counts[status] += 1
        counts["missing_labels"] += missing
        counts["invalid_labels"] += bad
        issues = []
        if status not in schema["statuses"]:
            issues.append("STATUS_TIDAK_VALID")
        elif status == "PENDING":
            issues.append("BELUM_DISETUJUI")
        elif status == "DEFERRED":
            issues.append("KASUS_DITUNDA")
            if not note:
                issues.append("CATATAN_TUNDA_WAJIB")
        if missing:
            issues.append("LABEL_KOSONG")
        if bad:
            issues.append("LABEL_TIDAK_VALID")
        if not reviewer:
            issues.append("REVIEWER_BELUM_DIISI")
        ready = status == "APPROVED" and not missing and not bad and bool(reviewer)
        counts["ready_narratives"] += int(ready)
        details.append({"review_id": key, "status": status, "missing_labels": missing, "invalid_labels": bad, "ready": ready, "issues": issues})
        labels_by_id[key] = labels
    complete = counts["ready_narratives"] == len(records)
    report = {
        "review_status": "COMPLETE_ALL_APPROVED" if complete else "AWAITING_REVIEW_OR_RESOLUTION",
        "ready_to_evaluate": complete,
        "reviewer": reviewer, "narratives_expected": len(records),
        "tokens_expected": sum(len(r["tokens"]) for r in records),
        "ready_narratives": counts["ready_narratives"],
        "pending_narratives": counts["PENDING"], "deferred_narratives": counts["DEFERRED"],
        "missing_labels": counts["missing_labels"], "invalid_labels": counts["invalid_labels"],
        "case_canonicalizations": canonicalizations, "invalid_label_details": invalid,
        "details": details, "blank_labels_converted_to_o": 0,
        "deferred_records_automatically_replaced_or_dropped": False,
        "neural_inference_executed": False, "metrics_computed": False,
    }
    return report, labels_by_id


def validate_file(workbook, data_dir):
    records, schema = load_locked_data(data_dir)
    reviewer, narratives, tokens = read_annotations(read_xlsx(workbook), records, schema)
    report, labels = validate_annotations(records, schema, reviewer, narratives, tokens)
    report["workbook_sha256"] = digest(workbook)
    return report, labels, records, narratives


def main():
    base = Path(__file__).resolve().parent.parent
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("workbook", type=Path)
    parser.add_argument("--data-dir", type=Path, default=base / "data")
    parser.add_argument("--output-dir", type=Path, default=base / "review_validation")
    parser.add_argument("--freeze", action="store_true")
    args = parser.parse_args()
    report, labels, records, narratives = validate_file(args.workbook, args.data_dir)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    dump(args.output_dir / "review_validation.json", report)
    print(json.dumps({k: report[k] for k in ("review_status", "ready_to_evaluate", "ready_narratives", "pending_narratives", "deferred_narratives", "missing_labels", "invalid_labels")}, indent=2))
    if args.freeze:
        if not report["ready_to_evaluate"]:
            raise SystemExit("Review belum lengkap; gold evaluasi tidak ditulis.")
        mapping = {r["review_id"]: r for r in read_json(args.data_dir / "selection_manifest.json")["selected_records"]}
        reviewed_at = datetime.now(timezone.utc).isoformat()
        rows = [{"id": mapping[r["review_id"]]["source_id"], "review_id": r["review_id"], "text": r["text"], "tokens": r["tokens"], "labels": labels[r["review_id"]], "reviewer": report["reviewer"], "review_note": narratives[r["review_id"]]["note"], "annotation_policy": "nara-role-guideline-0.2", "reviewed_at_utc": reviewed_at, "split": "test", "source": "REANNOTATED_HISTORICAL_TEST"} for r in records]
        target = args.output_dir / "test_reviewed_policy02.jsonl"
        with target.open("x", encoding="utf-8") as stream:
            for row in rows:
                stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
        dump(args.output_dir / "frozen_test_manifest.json", {"narratives": len(rows), "tokens": sum(len(r["tokens"]) for r in rows), "gold_sha256": digest(target), "workbook_sha256": report["workbook_sha256"], "reviewer": report["reviewer"], "reviewed_at_utc": reviewed_at, "candidate_lock_sha256": digest(args.data_dir / "candidate_lock.json"), "not_for_training_or_checkpoint_selection": True})


if __name__ == "__main__":
    main()

```


### nara_token_metrics.py

```python
"""Metrik token dan span flat; diturunkan dari audit terdahulu tanpa mengubah definisi.

Sumber SHA-256: c101387293b8f1ab60331880adb1badfcd001a0701e05a619f0c33dc00062ba2
"""

from collections import Counter

LABELS = (
    "O", "ACCIDENT-EVENT", "AFFECTEDOBJECTS-ARG", "DEATHVICTIM-ARG",
    "EARTHQUAKE-EVENT", "FALSE-EVENT", "FIRE-EVENT", "FLOOD-EVENT",
    "INFORMATION-ARG", "OFFICER-ARG", "PLACE-ARG", "REASON-ARG",
    "STREET-ARG", "TIME-ARG", "WOUNDVICTIM-ARG",
)



def divide(a, b):
    return a / b if b else 0.0



def flat_spans(labels):
    """Span kontigu label identik; O diabaikan. Tidak menafsirkan BIO."""
    spans = set()
    start = 0
    while start < len(labels):
        end = start + 1
        while end < len(labels) and labels[end] == labels[start]:
            end += 1
        if labels[start] != "O":
            spans.add((labels[start], start, end))
        start = end
    return spans



def recompute(rows):
    index = {label: i for i, label in enumerate(LABELS)}
    matrix = [[0] * len(LABELS) for _ in LABELS]
    span_counts = Counter()
    used = 0
    for row in rows:
        for gold, pred in zip(row["gold_labels"], row["predicted_labels"]):
            matrix[index[gold]][index[pred]] += 1
        if not row["coverage"]["truncated"]:
            used += 1
            gold, pred = flat_spans(row["gold_labels"]), flat_spans(row["predicted_labels"])
            span_counts["tp"] += len(gold & pred)
            span_counts["fp"] += len(pred - gold)
            span_counts["fn"] += len(gold - pred)
    total = sum(map(sum, matrix))
    by_label = {}
    for i, label in enumerate(LABELS):
        tp = matrix[i][i]
        gold_total = sum(matrix[i])
        pred_total = sum(row[i] for row in matrix)
        precision, recall = divide(tp, pred_total), divide(tp, gold_total)
        by_label[label] = {
            "precision": precision, "recall": recall,
            "f1-score": divide(2 * precision * recall, precision + recall),
            "support": gold_total, "predicted_tokens": pred_total,
        }
    roles = [by_label[label] for label in LABELS[1:]]
    role_tp = sum(matrix[i][i] for i in range(1, len(LABELS)))
    pred_roles = sum(row["predicted_tokens"] for row in roles)
    gold_roles = sum(row["support"] for row in roles)
    micro_precision, micro_recall = divide(role_tp, pred_roles), divide(role_tp, gold_roles)
    token_metrics = {
        "accuracy": divide(sum(matrix[i][i] for i in range(len(LABELS))), total),
        "precision_macro_no_o": sum(x["precision"] for x in roles) / 14,
        "recall_macro_no_o": sum(x["recall"] for x in roles) / 14,
        "macro_f1_no_o": sum(x["f1-score"] for x in roles) / 14,
        "micro_f1_no_o": divide(2 * micro_precision * micro_recall, micro_precision + micro_recall),
        "weighted_f1_no_o": divide(sum(x["f1-score"] * x["support"] for x in roles), gold_roles),
        "predicted_non_o_rate": divide(pred_roles, total),
        "gold_non_o_rate": divide(gold_roles, total),
    }
    sp, sr = divide(span_counts["tp"], span_counts["tp"] + span_counts["fp"]), divide(span_counts["tp"], span_counts["tp"] + span_counts["fn"])
    span_metrics = {
        "precision": sp, "recall": sr, "f1": divide(2 * sp * sr, sp + sr),
        "true_positive_spans": span_counts["tp"],
        "false_positive_spans": span_counts["fp"],
        "false_negative_spans": span_counts["fn"],
        "records_used": used, "records_excluded_due_to_truncation": len(rows) - used,
    }
    pairs = sorted(
        ({"gold": a, "prediction": b, "count": matrix[i][j]}
         for i, a in enumerate(LABELS) for j, b in enumerate(LABELS)
         if i != j and matrix[i][j]),
        key=lambda x: (-x["count"], x["gold"], x["prediction"]),
    )
    return {
        "narratives": len(rows), "token_count": total,
        "token_metrics": token_metrics, "span_metrics": span_metrics,
        "by_label": by_label, "confusion_matrix": matrix,
        "gold_o_predicted_non_o": sum(matrix[0][1:]),
        "gold_non_o_predicted_o": sum(row[0] for row in matrix[1:]),
        "top_confusions": pairs[:15],
    }



```


### evaluate_reviewed_nara.py

```python
#!/usr/bin/env python3
"""Nilai prediksi run tetap terhadap review manusia; tanpa training/inferensi.

--scope approved-subset menghasilkan analisis parsial bila ada DEFERRED.
--scope full80 tetap menolak evaluasi bila belum semua 80 disetujui.
Validator dan workbook sumber tidak diubah. Kasus DEFERRED tetap tercatat.
"""
from __future__ import annotations

import argparse
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
from pathlib import Path
import zipfile

from validate_review import validate_file, read_json, dump, digest
from nara_token_metrics import LABELS, recompute

RUN_ID = "20261018T001122_607337Z"
REPORT_NAME = f"SLR09_NARA_Hasil_FineTune_{RUN_ID}.zip"
REPORT_SHA = "879b3205419a9897075145a64c94e29c1f3b81b558af170a8181f55fdfa63bfb"


def jsonl(raw):
    if isinstance(raw, bytes):
        raw = raw.decode("utf-8")
    return [json.loads(line) for line in raw.splitlines() if line.strip()]


def write_jsonl(path, rows):
    Path(path).write_text("".join(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n" for r in rows), encoding="utf-8")


def write_csv(path, rows):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def evaluate(workbook, data_dir, results_zip, historical_gold, output, *, scope="approved-subset"):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    validation, reviewed_labels, records, narratives = validate_file(workbook, data_dir)
    dump(output / "review_validation.json", validation)
    if scope not in ("approved-subset", "full80"):
        raise ValueError("Scope tidak dikenal")
    if scope == "full80" and not validation["ready_to_evaluate"]:
        raise ValueError("Evaluasi penuh 80 ditolak: review masih memiliki kasus belum disetujui")
    ready = {r["review_id"] for r in validation["details"] if r["ready"]}
    if not ready:
        raise ValueError("Tidak ada narasi APPROVED lengkap untuk dianalisis")
    if any(d["status"] == "APPROVED" and not d["ready"] for d in validation["details"]):
        raise ValueError("Ada APPROVED yang belum memenuhi label/reviewer; selesaikan input terlebih dahulu")
    if digest(results_zip) != REPORT_SHA:
        raise ValueError("ZIP prediksi berbeda dari run tetap yang ditetapkan")
    selection = read_json(Path(data_dir) / "selection_manifest.json")
    locked = read_json(Path(data_dir) / "candidate_lock.json")
    mapping = {r["review_id"]: r for r in selection["selected_records"]}
    old_corpus = {r["id"]: r for r in jsonl(Path(historical_gold).read_bytes())}
    public = {r["review_id"]: r for r in records}
    approved_records = [r for r in records if r["review_id"] in ready]
    omitted = [{"review_id": r["review_id"], "source_id": mapping[r["review_id"]]["source_id"], "status": narratives[r["review_id"]]["status"], "tokens": len(r["tokens"]), "text": r["text"], "review_note": narratives[r["review_id"]]["note"], "temporary_labels_not_used_for_scoring": True} for r in records if r["review_id"] not in ready]
    checks = []
    def check(name, ok):
        if not ok:
            raise ValueError("Pemeriksaan gagal: " + name)
        checks.append({"check": name, "passed": True})
    check("fixed report SHA-256", digest(results_zip) == REPORT_SHA)
    provenance = read_json(Path(historical_gold).parent / "corpus_provenance.json")
    check("historical reference copy SHA-256", digest(historical_gold) == provenance["selected_jsonl_sha256"])
    check("historical source CSV matches frozen selection", provenance["csv_sha256"] == selection["source_csv_sha256"])
    check("stable source IDs unique and all test", len(mapping) == 80 and len({r["source_id"] for r in mapping.values()}) == 80 and all(r["original_nara_split"] == "test" for r in mapping.values()))
    predicted, metrics, source_metrics, changes = {}, {}, {}, Counter()
    gold_by_review = {}
    with zipfile.ZipFile(results_zip) as archive:
        check("report ZIP CRC", archive.testzip() is None)
        state = json.loads(archive.read("run_state.json"))
        exported = json.loads(archive.read("model_export_manifest.json"))
        check("candidate run fixed before reannotation", state["run_id"] == exported["run_id"] == locked["candidate_run_id"] == RUN_ID)
        check("reported baseline/candidate hash consistency", state["baseline_weights_sha256"] == locked["source_weights_sha256_reported"] == exported["source_weights_sha256"] and exported["new_weights_sha256"] == locked["candidate_weights_sha256_reported"])
        for model in ("baseline", "finetuned"):
            raw = jsonl(archive.read(model + "_historical_test/predictions.jsonl"))
            index = {r["id"]: r for r in raw}
            check(model + ": 1000 unique saved test predictions", len(raw) == len(index) == 1000)
            revised_rows, original_rows = [], []
            for record in approved_records:
                key = record["review_id"]
                source_id = mapping[key]["source_id"]
                row, original = index[source_id], old_corpus[source_id]
                if row["processed_tokens"] != record["tokens"] or original["tokens"] != record["tokens"] or row["text"] != record["text"]:
                    raise ValueError("Teks/token tidak sama: " + key)
                if row["gold_labels"] != original["labels"]:
                    raise ValueError("Gold historis berbeda dari salinan sumber: " + key)
                if row["coverage"]["truncated"] or row["coverage"]["partial_last_word"]:
                    raise ValueError("Subset mengandung narasi terpotong: " + key)
                if not (len(row["predicted_labels"]) == len(reviewed_labels[key]) == len(record["tokens"])) or not set(row["predicted_labels"]) <= set(LABELS):
                    raise ValueError("Bentuk label tidak sama: " + key)
                original_rows.append(row)
                revised_rows.append({**row, "gold_labels": reviewed_labels[key]})
                gold_by_review[key] = row["gold_labels"]
                if model == "baseline":
                    changes.update((a, b) for a, b in zip(row["gold_labels"], reviewed_labels[key]) if a != b)
            check(model + ": input and historical gold match source; complete token coverage", len(revised_rows) == len(ready))
            metrics[model] = {"reviewed_gold": recompute(revised_rows), "historical_gold_same_subset": recompute(original_rows)}
            predicted[model] = {r["id"]: r for r in revised_rows}
            source_metrics[model] = json.loads(archive.read(model + "_historical_test/metrics.json"))["token_metrics"]
    scored = len(approved_records)
    token_count = sum(len(r["tokens"]) for r in approved_records)
    partial = scored != len(records)
    support = dict(Counter(label for r in approved_records for label in reviewed_labels[r["review_id"]]))
    role_narratives = {label: sum(label in reviewed_labels[r["review_id"]] for r in approved_records) for label in LABELS}
    reference_rows, token_errors, record_errors = [], [], []
    for r in approved_records:
        key, source_id = r["review_id"], mapping[r["review_id"]]["source_id"]
        gold = reviewed_labels[key]
        base = predicted["baseline"][source_id]["predicted_labels"]
        fine = predicted["finetuned"][source_id]["predicted_labels"]
        reference_rows.append({"review_id": key, "source_id": source_id, "text": r["text"], "tokens": r["tokens"], "labels": gold, "reviewer": validation["reviewer"], "status": "APPROVED", "policy": "nara-role-guideline-0.2", "scope": "APPROVED_SUBSET_REFERENCE" if partial else "FULL_80_APPROVED_REFERENCE", "note": narratives[key]["note"]})
        base_wrong, fine_wrong = sum(a != b for a, b in zip(gold, base)), sum(a != b for a, b in zip(gold, fine))
        record_errors.append({"review_id": key, "source_id": source_id, "tokens": len(gold), "baseline_wrong_tokens": base_wrong, "finetuned_wrong_tokens": fine_wrong, "fewer_errors_after_finetune": base_wrong - fine_wrong, "text": r["text"], "review_note": narratives[key]["note"]})
        for i, (token, g, b, f) in enumerate(zip(r["tokens"], gold, base, fine), 1):
            if f != g or b != g:
                token_errors.append({"review_id": key, "source_id": source_id, "position": i, "token": token, "reviewed_gold": g, "historical_gold": gold_by_review[key][i-1], "baseline_prediction": b, "finetuned_prediction": f, "baseline_correct": g == b, "finetuned_correct": g == f})
    write_jsonl(output / "approved_reference_for_analysis.jsonl", reference_rows)
    write_csv(output / "narrative_errors.csv", record_errors)
    if token_errors:
        write_csv(output / "token_errors.csv", token_errors)
    label_table = []
    for label in LABELS:
        b, f = (metrics[m]["reviewed_gold"]["by_label"][label] for m in ("baseline", "finetuned"))
        label_table.append({"label": label, "gold_tokens": f["support"], "gold_narratives": role_narratives[label], "baseline_precision": b["precision"], "baseline_recall": b["recall"], "baseline_f1": b["f1-score"], "finetuned_precision": f["precision"], "finetuned_recall": f["recall"], "finetuned_f1": f["f1-score"], "f1_delta_percentage_points": 100 * (f["f1-score"] - b["f1-score"])})
    write_csv(output / "per_label.csv", label_table)
    write_csv(output / "historical_to_reviewed_changes.csv", [{"historical_label": a, "reviewed_label": b, "tokens_changed": count} for (a, b), count in changes.most_common()])
    write_csv(output / "omitted_cases.csv", [{"review_id": r["review_id"], "source_id": r["source_id"], "status": r["status"], "tokens": r["tokens"], "text": r["text"], "review_note": r["review_note"]} for r in omitted] or [{"review_id": "", "source_id": "", "status": "NONE_OMITTED", "tokens": 0, "text": "", "review_note": ""}])
    for model in ("baseline", "finetuned"):
        with (output / (model + "_confusion_matrix.csv")).open("w", encoding="utf-8-sig", newline="") as stream:
            writer = csv.writer(stream)
            writer.writerow(["gold / prediction", *LABELS])
            writer.writerows([[label, *row] for label, row in zip(LABELS, metrics[model]["reviewed_gold"]["confusion_matrix"])])
    result = {
        "schema": "silar-nara-revised-reference-evaluation-0.1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "evaluation_scope": "PARTIAL_APPROVED_SUBSET_NOT_FULL_80" if partial else "FULL_80_APPROVED",
        "requested_scope": scope,
        "review_validation_full_80_ready": validation["ready_to_evaluate"],
        "validation_summary": {"missing_labels": validation["missing_labels"], "invalid_labels": validation["invalid_labels"], "case_canonicalizations": len(validation["case_canonicalizations"])},
        "coverage": {"sample_narratives": len(records), "sample_tokens": sum(len(r["tokens"]) for r in records), "scored_narratives": scored, "scored_tokens": token_count, "omitted_narratives": len(omitted), "omitted_tokens": sum(r["tokens"] for r in omitted), "scored_source_groups": dict(Counter(mapping[r["review_id"]]["source_group"] for r in approved_records))},
        "reviewer": validation["reviewer"], "omitted_cases": omitted,
        "gold_label_support": support, "gold_narrative_support": role_narratives,
        "historical_to_reviewed_tokens_changed": sum(changes.values()),
        "historical_to_reviewed_change_pairs": [{"old": a, "new": b, "tokens": count} for (a, b), count in changes.most_common()],
        "metrics": metrics, "historical_test_1000_unchanged": source_metrics,
        "predictions_source": {"run_id": RUN_ID, "report_zip_sha256": REPORT_SHA, "reused_saved_predictions": True, "fresh_neural_inference_executed": False, "training_executed": False, "weights_independently_verified_here": False, "reported_baseline_weights_sha256": locked["source_weights_sha256_reported"], "reported_finetuned_weights_sha256": locked["candidate_weights_sha256_reported"]},
        "review_source_sha256": digest(workbook),
        "data_lock_sha256": digest(Path(data_dir) / "data_lock.json"),
        "metric_definition": "Accuracy over all 15 labels; macro/micro/weighted F1 over fixed 14 non-O labels; O-related errors still counted; exact contiguous flat spans, not BIO; source-token first-wordpiece projection.",
        "checks": checks,
        "full_80_frozen_gold_created": False,
        "uploaded_review_workbook_modified": False,
        "deferred_status_automatically_changed": False,
        "new_adapter_field_references_created": False,
        "sena_or_production_fusion_evaluated": False,
        "production_model_selected": False,
    }
    check("all scored counts agree", all(metrics[m][p]["narratives"] == scored and metrics[m][p]["token_count"] == token_count for m in metrics for p in metrics[m]))
    dump(output / "evaluation.json", result)
    return result


def plot(result, output):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    output = Path(output)
    coverage = result["coverage"]
    for model, title in (("baseline", "NARA v1"), ("finetuned", "NARA fine-tuned")):
        matrix = np.array(result["metrics"][model]["reviewed_gold"]["confusion_matrix"])
        fig, ax = plt.subplots(figsize=(13, 10))
        im = ax.imshow(matrix, cmap="Blues")
        for i in range(15):
            for j in range(15):
                if matrix[i, j]:
                    ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=8, color="white" if matrix[i, j] > matrix.max() / 2 else "black")
        ax.set_xticks(range(15), LABELS, rotation=60, ha="right", fontsize=8)
        ax.set_yticks(range(15), LABELS, fontsize=8)
        ax.set(xlabel="Predicted label", ylabel="Reviewed gold label", title=f"{title}: revised-policy gold, {coverage['scored_narratives']}/{coverage['sample_narratives']} approved narratives")
        fig.colorbar(im, ax=ax, shrink=.7)
        fig.tight_layout()
        fig.savefig(output / (model + "_confusion_matrix.png"), dpi=180)
        plt.close(fig)
    rows = result["metrics"]
    fig, ax = plt.subplots(figsize=(11, 8))
    y = np.arange(14)
    b = [rows["baseline"]["reviewed_gold"]["by_label"][label]["f1-score"] * 100 for label in LABELS[1:]]
    f = [rows["finetuned"]["reviewed_gold"]["by_label"][label]["f1-score"] * 100 for label in LABELS[1:]]
    ax.barh(y - .18, b, .36, label="NARA v1", color="#8C9EAF")
    ax.barh(y + .18, f, .36, label="Fine-tuned", color="#315E8C")
    ax.set_yticks(y, [f"{label} (n={result['gold_label_support'][label]} tokens)" for label in LABELS[1:]], fontsize=8)
    ax.invert_yaxis()
    ax.set(xlim=(0, 105), xlabel="Token F1 (%)", title=f"Reviewed gold: {coverage['scored_narratives']} approved narratives; support shown per role")
    handles, legend_labels = ax.get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="lower center", bbox_to_anchor=(.67, .005), ncol=2, frameon=False)
    ax.grid(axis="x", alpha=.15)
    ax.set_axisbelow(True)
    fig.tight_layout(rect=(0, .06, 1, 1))
    fig.savefig(output / "f1_per_label.png", dpi=180)
    plt.close(fig)


def main():
    base = Path(__file__).resolve().parent.parent
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--workbook", type=Path, default=base / "sources/SLR09_NARA_Review_Uji_80_Reviewed.xlsx")
    parser.add_argument("--data-dir", type=Path, default=base / "data")
    parser.add_argument("--results-zip", type=Path, default=base / "sources" / REPORT_NAME)
    parser.add_argument("--historical-gold", type=Path, default=base / "sources/historical_test_original_labels.jsonl")
    parser.add_argument("--output", type=Path, default=base / "results")
    parser.add_argument("--scope", choices=["approved-subset", "full80"], required=True)
    parser.add_argument("--plots", action="store_true")
    args = parser.parse_args()
    result = evaluate(args.workbook, args.data_dir, args.results_zip, args.historical_gold, args.output, scope=args.scope)
    if args.plots:
        plot(result, args.output)
    print(json.dumps({"scope": result["evaluation_scope"], "coverage": result["coverage"], "baseline_macro_f1": result["metrics"]["baseline"]["reviewed_gold"]["token_metrics"]["macro_f1_no_o"], "finetuned_macro_f1": result["metrics"]["finetuned"]["reviewed_gold"]["token_metrics"]["macro_f1_no_o"], "fresh_neural_inference": False}, indent=2))


if __name__ == "__main__":
    main()

```


### write_report.py

```python
#!/usr/bin/env python3
"""Laporan evaluasi dari angka aktual; tidak mengubah anotasi/prediksi."""
import csv
import json
from pathlib import Path


def pct(x):
    return f"{100*x:.2f}%".replace(".", ",")


def pp(x):
    return f"{100*x:+.2f}".replace(".", ",")


def table(headers, rows):
    return "\n".join(["| " + " | ".join(headers) + " |", "| " + " | ".join("---" for _ in headers) + " |", *["| " + " | ".join(str(v).replace("|", "\\|") for v in row) + " |" for row in rows]])


def write_report(result, output, results_dir):
    d, c = result, result["coverage"]
    b, f = (d["metrics"][m]["reviewed_gold"] for m in ("baseline", "finetuned"))
    old_b, old_f = (d["metrics"][m]["historical_gold_same_subset"] for m in ("baseline", "finetuned"))
    with (Path(results_dir) / "narrative_errors.csv").open(encoding="utf-8-sig") as stream:
        record_errors = list(csv.DictReader(stream))
    improved = sum(int(r["fewer_errors_after_finetune"]) > 0 for r in record_errors)
    regressed = sum(int(r["fewer_errors_after_finetune"]) < 0 for r in record_errors)
    same = len(record_errors) - improved - regressed
    lines = [
        "# SLR09 — Evaluasi NARA terhadap acuan revisi v0.1", "",
        "**Workflow:** validasi Excel → tentukan cakupan sesuai status review → cocokkan prediksi run tetap → hitung token/span dan matriks → analisis hasil.", "",
        f"**Status: {d['evaluation_scope']}.** Sampel awal {c['sample_narratives']} narasi/{c['sample_tokens']} token. "
        f"Yang dinilai: **{c['scored_narratives']} narasi/{c['scored_tokens']} token**, dengan {c['omitted_narratives']} narasi/{c['omitted_tokens']} token belum disetujui. "
        "Hasil ini tidak disebut sebagai evaluasi penuh 80 narasi.", "",
        "## Hasil utama", "",
        "Prediksi kandidat fine-tune lebih sesuai dengan acuan revisi pada subset yang disetujui. "
        f"Macro F1 non-O meningkat dari **{pct(b['token_metrics']['macro_f1_no_o'])} menjadi {pct(f['token_metrics']['macro_f1_no_o'])}**, "
        f"atau **{pp(f['token_metrics']['macro_f1_no_o']-b['token_metrics']['macro_f1_no_o'])} poin persentase**. "
        "Ini merupakan evaluasi ulang prediksi yang telah tersimpan, tanpa melatih model atau menjalankan inferensi neural baru.", "",
        table(["Metrik terhadap acuan revisi", "NARA v1", "Fine-tune", "Selisih (poin persentase)"], [
            *[[label, pct(b["token_metrics"][key]), pct(f["token_metrics"][key]), pp(f["token_metrics"][key]-b["token_metrics"][key])] for label, key in [("Akurasi token", "accuracy"), ("Precision macro non-O", "precision_macro_no_o"), ("Recall macro non-O", "recall_macro_no_o"), ("Macro F1 non-O", "macro_f1_no_o"), ("Micro F1 non-O", "micro_f1_no_o"), ("Weighted F1 non-O", "weighted_f1_no_o")]],
            ["F1 span tepat", pct(b["span_metrics"]["f1"]), pct(f["span_metrics"]["f1"]), pp(f["span_metrics"]["f1"]-b["span_metrics"]["f1"])],
        ]), "",
        "Macro F1 memakai seluruh 14 label non-O dengan bobot sama per label. Akurasi token memasukkan O. "
        "Kesalahan dari/ke O tetap memengaruhi precision dan recall role. F1 span menuntut label dan batas token tepat untuk span flat yang kontigu; format ini tidak dianggap BIO. "
        f"Semua {c['scored_tokens']} token sumber tercakup dan tidak ada narasi dinyatakan terpotong dalam metadata inferensi. "
        "Teks sumber yang memang berhenti di tengah kalimat tetap dinilai sebagaimana tertulis, terpisah dari truncation oleh batas input model.", "",
        "## Validasi review", "",
        f"Reviewer: **{d['reviewer']}**. Pada {c['sample_tokens']} token dalam Excel: "
        f"label kosong {d['validation_summary']['missing_labels']}, label tidak dikenal {d['validation_summary']['invalid_labels']}, "
        f"dan normalisasi huruf {d['validation_summary']['case_canonicalizations']}. "
        "ID, teks, token sumber, posisi token, dan daftar label cocok dengan paket uji terkunci. "
        "Workbook pengguna tidak diubah.", "",
        f"Dari {c['sample_narratives']} narasi, {c['scored_narratives']} berstatus APPROVED dan {c['omitted_narratives']} masih DEFERRED. "
        "Validator evaluasi penuh tetap menyatakan belum siap. Analisis ini membaca hanya acuan yang telah disetujui dan mencatat kasus yang tidak dinilai; "
        "tidak membuat gold penuh 80, tidak mengganti kasus dengan sampel lain, dan tidak mengubah DEFERRED menjadi APPROVED.", "",
        table(["ID review", "ID sumber", "Status", "Token", "Catatan reviewer"], [[r["review_id"], r["source_id"], r["status"], r["tokens"], r["review_note"]] for r in d["omitted_cases"]] or [["—", "—", "Semua selesai", 0, "—"]]), "",
    ]
    for r in d["omitted_cases"]:
        lines += ["> " + r["text"], "", "Label sementara kasus ini tidak dipakai untuk skor. Status DEFERRED dipertahankan sesuai file yang dikirim; belum ada keputusan pengguna yang mengubahnya.", ""]
    lines += [
        "Proporsi kelompok sumber pada subset yang dinilai: " + ", ".join(f"{k} {v}" for k, v in sorted(c["scored_source_groups"].items())) + ". "
        "Kelompok sumber adalah asal pengumpulan narasi, bukan kategori kejadian final SENA. "
        "Keputusan menunda kasus ambigu dapat memengaruhi karakter subset; cakupan dan pengecualiannya harus tetap dilaporkan.", "",
        "## Dampak perubahan anotasi pada narasi yang sama", "",
        "Untuk memeriksa pengaruh acuan, kedua prediksi tetap juga dinilai terhadap label historis pada **subset narasi yang sama**:", "",
        table(["Acuan pada subset yang sama", "Macro F1 NARA v1", "Macro F1 fine-tune"], [
            ["Historis", pct(old_b["token_metrics"]["macro_f1_no_o"]), pct(old_f["token_metrics"]["macro_f1_no_o"])],
            ["Review pedoman 0.2", pct(b["token_metrics"]["macro_f1_no_o"]), pct(f["token_metrics"]["macro_f1_no_o"])],
        ]), "",
        f"Review mengubah label **{d['historical_to_reviewed_tokens_changed']} dari {c['scored_tokens']} token** "
        f"({pct(d['historical_to_reviewed_tokens_changed']/c['scored_tokens'])}). "
        f"Proporsi non-O pada acuan naik dari {pct(old_b['token_metrics']['gold_non_o_rate'])} menjadi {pct(b['token_metrics']['gold_non_o_rate'])}. "
        "Perubahan ini membalik urutan performa dua model pada subset yang sama. Ini merupakan bukti langsung bahwa ketidakcocokan anotasi memengaruhi penilaian pada subset ini. "
        "Bukti tersebut tidak menetapkan bahwa seluruh penurunan pada test historis disebabkan oleh perubahan anotasi, dan tidak menghasilkan estimasi untuk 1.000 acuan revisi.", "",
        table(["Perubahan label historis → revisi", "Token"], [[r["old"] + " → " + r["new"], r["tokens"]] for r in d["historical_to_reviewed_change_pairs"][:8]]), "",
        "Test historis 1.000 narasi tetap memiliki macro F1 **" + pct(d["historical_test_1000_unchanged"]["baseline"]["macro_f1_no_o"]) + " → " + pct(d["historical_test_1000_unchanged"]["finetuned"]["macro_f1_no_o"]) + "**. "
        "Hasil historis tersebut tetap dipertahankan dalam pelaporan bersama hasil revisi; keduanya mengukur kesesuaian terhadap acuan yang berbeda.", "",
        "## Rincian per label", "",
        table(["Label", "Token acuan", "Narasi dengan label", "F1 NARA v1", "F1 fine-tune"], [[label, f["by_label"][label]["support"], d["gold_narrative_support"][label], pct(b["by_label"][label]["f1-score"]), pct(f["by_label"][label]["f1-score"])] for label in f["by_label"]]), "",
        "Dukungan setiap label dilaporkan pada tabel; denominator macro F1 tetap seluruh 14 non-O. "
        f"WOUND memiliki F1 {pct(f['by_label']['WOUNDVICTIM-ARG']['f1-score'])}, dengan dukungan "
        f"**{f['by_label']['WOUNDVICTIM-ARG']['support']} token pada {d['gold_narrative_support']['WOUNDVICTIM-ARG']} narasi**. "
        "Ini belum membuktikan penguasaan seluruh variasi luka/kondisi fisik dan tidak membatalkan kegagalan kasus sesak nafas pada validasi terdahulu. "
        "Label REASON masih lemah: F1 **" + pct(f["by_label"]["REASON-ARG"]["f1-score"]) + "**, dengan " + str(f["by_label"]["REASON-ARG"]["support"]) + " token acuan. "
        "INFORMATION memiliki F1 **" + pct(f["by_label"]["INFORMATION-ARG"]["f1-score"]) + "** dan FALSE-EVENT **" + pct(f["by_label"]["FALSE-EVENT"]["f1-score"]) + "**.", "",
        "## Kesalahan yang masih terlihat", "",
        f"Berdasarkan jumlah token salah per narasi: **{improved} membaik, {regressed} memburuk, dan {same} tetap**. "
        "Ukuran ini adalah perbandingan jumlah kesalahan token, bukan skor exact seluruh narasi.", "",
        table(["Acuan → prediksi fine-tune", "Token salah"], [[r["gold"] + " → " + r["prediction"], r["count"]] for r in f["top_confusions"][:8]]), "",
        "Pada INFORMATION, sebagian informasi masih hilang menjadi O dan sebagian token O justru diberi INFORMATION. "
        "Ini mendukung pemeriksaan batas frasa dan arti hasil ekstraksi pada integrasi berikutnya. "
        "Token berlabel INFORMATION tidak otomatis membuktikan nilai field impacts sudah benar.", "",
        f"{regressed} narasi dengan jumlah kesalahan token bertambah:", "",
        table(["ID review", "Token salah NARA v1", "Token salah fine-tune", "Catatan reviewer"], [[r["review_id"], r["baseline_wrong_tokens"], r["finetuned_wrong_tokens"], r["review_note"]] for r in record_errors if int(r["fewer_errors_after_finetune"]) < 0]), "",
        "## Batas kesimpulan dan langkah berikutnya", "",
        "1. Ini evaluasi parsial pada 79 narasi yang disetujui dari sampel test historis yang pernah dievaluasi. Tidak disebut test baru yang sepenuhnya belum pernah digunakan.",
        "2. Sampling ditetapkan tanpa prediksi, skor kesalahan, atau label role historis. Review disusun tanpa menampilkan label lama dan prediksi pada workbook.",
        "3. Data test NARA terpisah dari seluruh train/validation NARA asli dan tidak dipakai fine-tune/pemilihan epoch. Keterpisahan gabungan terhadap pelatihan SENA tidak dibuktikan oleh paket ini.",
        "4. Hasil adalah token dan span NARA. Tujuh field A4.22, SENA, fusion, foto, threshold, dan deployment tidak dievaluasi ulang di sini.",
        "5. Prediksi dibaca dari run Colab tetap. Pemeriksaan langsung bobot ekspor baru belum dilakukan; identitas bobot mengikuti manifest run yang hash arsip laporannya diverifikasi.",
        "6. Sampel kecil dan satu reviewer membatasi kesimpulan luas. Dukungan role langka harus dibaca bersama F1, terutama WOUND.", "",
        "Hasil ini mendukung kandidat fine-tune sebagai model yang lebih selaras dengan pedoman revisi pada subset tersebut. "
        "Status model produksi belum ditetapkan. Tahap selanjutnya yang disarankan adalah pemeriksaan integrasi kandidat NARA dengan adapter tujuh field, "
        "terutama batas impacts, pihak, waktu, dan penyebab, menggunakan artefak diagnostik yang telah tersedia. "
        "UJI-019 tetap terbuka sampai ada keputusan reviewer; hasil 80 narasi penuh baru dapat dihitung setelahnya.", "",
        "## Artefak dan reproduksi", "",
        "Notebook `SLR09_NARA_Evaluasi_Acuan_Revisi_v0.1.ipynb` berisi kode evaluasi lengkap. "
        "Notebook memakai snapshot Excel review dan data terkunci, lalu membaca ZIP hasil prediksi yang sudah ada di Drive. "
        "Tidak memerlukan bobot model atau GPU. Hasil pada folder results sudah dihitung dalam pemeriksaan ini; menjalankan notebook bersifat reproduksi.", "",
        "Paket kode lengkap menyertakan validator asli tanpa perubahan, metrik token/span, kode evaluasi dan laporan, snapshot input, "
        "provenance, tabel kesalahan, dua confusion matrix, grafik F1 per label, serta pemeriksaan independen metrik. "
        "Label review dan status dalam workbook unggahan dipertahankan byte-identik.", "",
        "SHA-256 Excel review: `" + d["review_source_sha256"] + "`.",
        "SHA-256 ZIP hasil run: `" + d["predictions_source"]["report_zip_sha256"] + "`.", "",
    ]
    Path(output).write_text("\n".join(lines), encoding="utf-8")


if __name__ == "__main__":
    base = Path(__file__).resolve().parent.parent
    result = json.loads((base / "results/evaluation.json").read_text())
    output = base / "SLR09_NARA_Laporan_Evaluasi_Acuan_Revisi_v0.1.md"
    write_report(result, output, base / "results")
    print(output.name)

```
